# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsf-rawnak/FlyRankAI-ML-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**The traffic totals are heavily right-skewed; content length isn't.** `impressions_90d` has a
mean of 5,200 but a median of just 731 — the average is being pulled up by a small number of
very high-traffic pages (p95 = 22,997, max = 517,715). `sessions_90d` shows the same shape
(mean 37 vs median 7). That's why both get `log1p`'d before becoming model features (section 1
of the leakage-check notebook) — a linear model fed the raw counts would be dominated by a
handful of outlier pages. `word_count`, by contrast, is close to normal (mean 3,108, median
2,878) — content length doesn't have the same long tail traffic does.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

local_path = Path("../../data/raw/content_refresh_anonymized.csv")
raw_url = "https://raw.githubusercontent.com/rsf-rawnak/FlyRankAI-ML-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(local_path) if local_path.exists() else pd.read_csv(raw_url)

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
for col in ["impressions_90d", "sessions_90d", "word_count", "engagement_rate",
            "days_since_last_update", "avg_position"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("mean vs median vs p95 vs max - the gap IS the tail:")
for col in ["impressions_90d", "sessions_90d", "word_count"]:
    s = df[col].dropna()
    print(f"  {col:20s} mean={s.mean():9.1f}  median={s.median():8.1f}  p95={s.quantile(.95):9.1f}  max={s.max():10.1f}")

mean vs median vs p95 vs max - the gap IS the tail:
  impressions_90d      mean=   5200.4  median=   731.0  p95=  22996.5  max=  517715.0
  sessions_90d         mean=     37.1  median=     7.0  p95=    166.0  max=    4345.0
  word_count           mean=   3107.8  median=  2877.0  p95=   6173.0  max=    9546.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Test 1 — "more engaged sessions means a page is safer from decline."**
Splitting on the top vs bottom quartile of `engagement_rate`, the decline rate barely moves
(53.3% vs 54.4%, corr = -0.013). **Verdict: FALSE.** Engagement on its own doesn't protect a page
— which matters, because it's the kind of metric that *feels* like it should.

**Test 2 — "staler content (longer since last update) declines more."**
Splitting on the median `days_since_last_update`, stale pages decline at 54.5% vs 52.3% for
fresher ones (corr = +0.081). The direction is right but the gap is small. **Verdict: MIXED** —
weakly true, not something to lean on alone.

**Test 3 — "worse average search position means a page declines more."**
Splitting on the median `avg_position` (excluding the 0 = "no data" rows), decline rates are
56.3% (worse position) vs 56.6% (better position) — statistically a wash, and if anything
pointing the *wrong* way (corr = -0.081). **Verdict: OPPOSITE (weak)** — the intuitive
assumption that bad rankings predict decline doesn't hold up in this slice on its own.

In [2]:
# Test 1: engagement_rate
q1, q3 = df["engagement_rate"].quantile([0.25, 0.75])
top_eng = df[df["engagement_rate"] >= q3]["is_declining_label"].mean()
bot_eng = df[df["engagement_rate"] <= q1]["is_declining_label"].mean()
print(f"Test 1 - engagement_rate: top quartile decline={top_eng:.3f}  bottom quartile decline={bot_eng:.3f}  corr={df['engagement_rate'].corr(df['is_declining_label']):.3f}")

# Test 2: days_since_last_update
med = df["days_since_last_update"].median()
stale = df[df["days_since_last_update"] >= med]["is_declining_label"].mean()
fresh = df[df["days_since_last_update"] < med]["is_declining_label"].mean()
print(f"Test 2 - days_since_last_update (median={med:.0f}d): stale decline={stale:.3f}  fresh decline={fresh:.3f}  corr={df['days_since_last_update'].corr(df['is_declining_label']):.3f}")

# Test 3: avg_position (0 = no data, exclude)
sub = df[df["avg_position"] > 0]
med_pos = sub["avg_position"].median()
worse = sub[sub["avg_position"] >= med_pos]["is_declining_label"].mean()
better = sub[sub["avg_position"] < med_pos]["is_declining_label"].mean()
print(f"Test 3 - avg_position (median={med_pos:.1f}, n={len(sub)}): worse-position decline={worse:.3f}  better-position decline={better:.3f}  corr={sub['avg_position'].corr(sub['is_declining_label']):.3f}")

Test 1 - engagement_rate: top quartile decline=0.533  bottom quartile decline=0.544  corr=-0.013
Test 2 - days_since_last_update (median=20d): stale decline=0.545  fresh decline=0.523  corr=0.081
Test 3 - avg_position (median=11.4, n=28795): worse-position decline=0.563  better-position decline=0.566  corr=-0.081


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's
assumption?*

**The baseline rule (`scripts/02_baseline_score.py`) weighs freshness at 0.30** — the second
heaviest of its four inputs — on the assumption that older-since-update content is steadily
worse off. `freshness_tier` breaks that assumption at the tail: decline rate climbs from 51.1%
(`0-30` days) to 61.1% (`91-180` days) as the rule expects, but then *drops back down* to 47.1%
for the stalest bucket (`181+` days, n=174). The middle bucket (`31-90`) is also thin (n=175)
and sits *above* the older bucket at 58.9% — not the clean monotonic staircase a fixed 0.30
weight assumes. **Verdict: MIXED.** The direction holds for the bulk of the data (the two buckets
holding 98.8% of rows) but breaks down exactly where a fixed rule would flag hardest — the very
stalest pages aren't reliably the worst off.

In [3]:
df["freshness_tier"] = df["freshness_tier"].fillna("unknown").astype(str)
tier_order = ["0-30", "31-90", "91-180", "181+"]
tier_stats = df.groupby("freshness_tier")["is_declining_label"].agg(["mean", "count"]).reindex(tier_order)
tier_stats.columns = ["decline_rate", "n_rows"]
print("Decline rate by freshness_tier (baseline rule weighs this bucket at 0.30):")
print(tier_stats.round(3))

Decline rate by freshness_tier (baseline rule weighs this bucket at 0.30):
                decline_rate  n_rows
freshness_tier                      
0-30                   0.511   20480
31-90                  0.589     175
91-180                 0.611    9171
181+                   0.471     174


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

No single hand-picked signal — not engagement, not staleness, not search position — reliably
tells a reviewer which pages are declining on its own; the strongest of the three (staleness)
still only nudges the odds by a few points. That's exactly why a learned model that weighs
several weak, tangled signals together (ML-03's framing) beats a fixed-weight rule here: the
fixed rule's 0.30 freshness weight is *right on average* but wrong at the tail, exactly where a
reviewer would trust it most. Practically: don't let any one flag — including the ones baked
into the baseline score — override a page's full profile without a second signal backing it up.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.